# Data Ingestion

### Document load

In [30]:
# pdf => document
import os
from langchain_community.document_loaders import PyPDFLoader

In [31]:
def load_all_pdfs():
    folder_path = "Data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path,filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1

    print("Total pdfs:",num_docs)
    print("Total pages:",len(all_docs)) # one object = one page
    return all_docs

In [32]:
all_pdf_documents = load_all_pdfs()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)


Total pdfs: 2
Total pages: 35


### chucks

In [33]:
# !pip install langchain_text_splitters

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [35]:
chunks = split_doc(all_pdf_documents)

In [36]:
len(chunks)

324

### Embedding

In [37]:
from sentence_transformers import SentenceTransformer

In [38]:
class EmbeddingManager:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        print(f"model loading....{self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"embedding dimensions =  {self.model.get_embedding_dimension()}")

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape: ",embeddings.shape)
        return embeddings     

In [39]:
embedding_manager = EmbeddingManager()

model loading....all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions =  384


### Vector Store

In [40]:
import chromadb
import uuid

In [41]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create Client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"Vector store collection for pdfs embeddings in RAG"}
        )

        print("Intialized the vector store with collection ",self.collection_name)
        print("docs in collection = ",self.collection.count())   

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            return ValueError("number of docs doesn't match num of embeddings")

        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_lenght"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embedding_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embedding_list
        )

        print("total documents add in vector store= ", len(documents_content))
        print("docs in collection= ",self.collection.count())            

In [42]:
vector_store = VectorStoreManager()

Intialized the vector store with collection  pdf_documents
docs in collection =  648


In [43]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape:  (324, 384)
total documents add in vector store=  324
docs in collection=  972


# Retrieval Pipeline

In [44]:
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=3, score_threshold=0.0):
        # query => embeddings
        query_embedding = embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1-distance

                if similarity_score >=  score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "metadata":metadata,
                        "document":document,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i+1
                    })
            print(f"retrieved document length {len(retrieved_docs)}")
        else:
            print("No document found")

        return retrieved_docs
            

In [50]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)
rag_retriever.retrieve("what is rag")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrieved document length 3


[{'id': 'doc_81245645-17c8-406c-a214-ec452313fab5',
  'metadata': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 0,
   'doc_index': 11,
   'moddate': '2024-03-28T00:54:45+00:00',
   'page_label': '1',
   'keywords': '',
   'author': '',
   'content_lenght': 288,
   'source': 'Data\\Research.pdf',
   'subject': '',
   'trapped': '/False',
   'creator': 'LaTeX with hyperref',
   'title': '',
   'total_pages': 21,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25'},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.42267584800720215,
  'similarity_score': 0.5773241519927979,
  'rank': 1},
 {'id': 'doc_a88b4883-7dee-

# Integrate  with LLMs

### Open AI - GPT

In [52]:
API_KEY_OPENAI="xyz"

In [54]:
#!pip install langchain-openai

In [55]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [64]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k)

    context = "/n".join([doc["document"]  for doc in results])  if  results else ""

    if not context:
        print("we found no relevant context for given query")

    # content + query
    prompt = f'''  use given content  to generate the answer for the query 
                    context:{context}
                    Query:{query}'''
    response = llm.invoke(prompt)

    return response

In [68]:
answer = generate_output("what is rag?",rag_retriever, llm)

## Groq

In [67]:
#!pip install langchain-groq 

In [83]:
from langchain_groq import ChatGroq

API_KEY_GROQ = "xyz"

llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    temperature=0.1,
    max_tokens=1024,
    model="openai/gpt-oss-120b",
)

In [84]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k)

    context = "/n".join([doc["document"]  for doc in results])  if  results else ""

    if not context:
        print("we found no relevant context for given query")

    # content + query
    prompt = f'''  use given content  to generate the answer for the query 
                    context:{context}
                    Query:{query}'''
    response = llm.invoke(prompt)

    return response

In [89]:
answer = generate_output("what is rag?", rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrieved document length 3


In [90]:
print(answer.content)

**RAG (Retrieval‑Augmented Generation)** is a hybrid approach that combines two core components:

1. **Retriever** – a module that searches an external knowledge source (e.g., a document corpus, database, or the web) and pulls the most relevant passages for a given query.  
2. **Generator** – a large language model (LLM) that takes the retrieved passages as additional context and produces a response that is both fluent and grounded in factual information.

The idea is to augment the generative capabilities of LLMs with up‑to‑date, domain‑specific, or otherwise hard‑to‑learn knowledge that the model alone might not contain. By conditioning generation on retrieved evidence, RAG reduces hallucinations and improves factual accuracy.

### How RAG works (high‑level flow)

1. **Input query** → passed to the retriever.  
2. **Retriever** returns a set of top‑k relevant documents/passages.  
3. **Generator** receives the original query plus the retrieved texts (often concatenated or encoded wit